# 案例二：幫值班人員先整理緊急訊息

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnychao/python-machine-learning-2026-student/blob/main/notebooks/ai_solution_practicum/02_rnn_message_story.ipynb)

值班人員收到大量短訊息，希望先標出可能與災害有關的內容，
再由人員依地點、時間與來源複核。

**情境問題：** 在其他設定都相同時，把 SimpleRNN 換成 GRU，
驗證集 F1 是否改變？

- baseline：Embedding + SimpleRNN。
- candidate：只把循環層換成同寬度的 GRU。
- 驗證邊界：文字向量器只在訓練資料上 adapt。
- [Kaggle 題目出處：Natural Language Processing with Disaster Tweets]
  (https://www.kaggle.com/competitions/nlp-getting-started)

執行資料由固定版本的公開鏡像直接讀入 Colab。


## 1. 匯入套件與固定設定


In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

SEED = 20260719
QUICK_MODE = True
EPOCHS = 3 if QUICK_MODE else 8
BATCH_SIZE = 64
MAX_TOKENS = 8_000
SEQUENCE_LENGTH = 40
DATA_URL = "https://huggingface.co/datasets/startificial/twitter-nlp/resolve/bb1c6925112f570ea0ca94c723ff8818cc642eed/train.csv"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. 直接讀取固定版本 CSV


In [ ]:
messages = pd.read_csv(DATA_URL)
required_columns = {"text", "target"}
missing_columns = required_columns - set(messages.columns)
assert not missing_columns, f"缺少欄位：{sorted(missing_columns)}"

messages = (
    messages.loc[:, ["text", "target"]]
    .dropna()
    .assign(
        text=lambda frame: frame["text"].astype(str),
        target=lambda frame: frame["target"].astype(int),
    )
)
assert set(messages["target"].unique()).issubset({0, 1})

print("資料筆數:", len(messages))
display(messages.head(5))
display(messages["target"].value_counts(normalize=True).rename("比例"))


## 3. 先切資料，再學文字表達

這一步是防止 data leakage 的重點：TextVectorization 只能看
訓練文字，不能先看驗證文字。


In [ ]:
train_text, validation_text, train_label, validation_label = train_test_split(
    messages["text"],
    messages["target"],
    test_size=0.2,
    random_state=SEED,
    stratify=messages["target"],
)

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)
vectorizer.adapt(
    tf.data.Dataset.from_tensor_slices(train_text.to_numpy()).batch(256)
)

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_text.to_numpy(), train_label.to_numpy())
)
train_ds = train_ds.shuffle(
    len(train_text),
    seed=SEED,
    reshuffle_each_iteration=False,
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

validation_ds = tf.data.Dataset.from_tensor_slices(
    (validation_text.to_numpy(), validation_label.to_numpy())
)
validation_ds = validation_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("訓練筆數:", len(train_text), "驗證筆數:", len(validation_text))


## 4. 建立公平比較函式

只允許 recurrent_kind 改變；Embedding、神經元數、dropout、
optimizer、epoch 與資料切分保持一致。


In [ ]:
def build_sequence_model(recurrent_kind: str) -> tf.keras.Model:
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    if recurrent_kind == "simple_rnn":
        recurrent_layer = tf.keras.layers.SimpleRNN(32)
    elif recurrent_kind == "gru":
        recurrent_layer = tf.keras.layers.GRU(32)
    else:
        raise ValueError(f"未知循環層：{recurrent_kind}")

    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(), dtype=tf.string),
            vectorizer,
            tf.keras.layers.Embedding(MAX_TOKENS, 32, mask_zero=True),
            recurrent_layer,
            tf.keras.layers.Dropout(0.2, seed=SEED),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


def evaluate_text_model(model) -> tuple[dict, np.ndarray]:
    probabilities = model.predict(validation_ds, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(int)
    labels = validation_label.to_numpy()
    metrics = {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
    }
    return metrics, probabilities


## 5. baseline：SimpleRNN


In [ ]:
baseline_model = build_sequence_model("simple_rnn")
baseline_history = baseline_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    verbose=2,
)
baseline_metrics, baseline_probabilities = evaluate_text_model(baseline_model)
baseline_metrics


## 6. candidate：只換成 GRU


In [ ]:
candidate_model = build_sequence_model("gru")
candidate_history = candidate_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    verbose=2,
)
candidate_metrics, candidate_probabilities = evaluate_text_model(candidate_model)
candidate_metrics


## 7. 比較前後


In [ ]:
comparison = pd.DataFrame(
    [baseline_metrics, candidate_metrics],
    index=["baseline_simple_rnn", "candidate_gru"],
)
display(comparison.style.format("{:.3f}"))

f1_change = candidate_metrics["f1"] - baseline_metrics["f1"]
chosen_version = "candidate_gru" if f1_change > 0 else "baseline_simple_rnn"
print(f"F1 改變：{f1_change:+.3f}")
print("依本次驗證 F1 暫選：", chosen_version)


## 8. 找出最需要人工看的訊息


In [ ]:
candidate_prediction = (candidate_probabilities >= 0.5).astype(int)
review_table = pd.DataFrame(
    {
        "text": validation_text.to_numpy(),
        "target": validation_label.to_numpy(),
        "prediction": candidate_prediction,
        "probability": candidate_probabilities,
    }
)
review_table["uncertainty"] = (review_table["probability"] - 0.5).abs()

print("判錯案例：")
display(
    review_table.loc[review_table["target"] != review_table["prediction"]]
    .sort_values("uncertainty")
    .head(10)
)
print("最不確定案例：")
display(review_table.sort_values("uncertainty").head(10))


## 9. 限制與人工介入

- 模型只讀短文字，沒有查證事件、地點、時間與發文者可信度。
- 反諷、轉述、比喻與拼字變形容易誤判。
- F1 只能描述這次驗證切分，不能推論真實災害處置成效。
- 高風險訊息不能由模型自動結案；低信心與涉及人身安全者都要人工複核。

請寫一句結論：這個模型適合做「排序提示」還是「自動判定」？為什麼？


## 10. 下載實驗紀錄與待複核清單


In [ ]:
review_path = Path("/content/rnn_message_review.csv")
review_table.sort_values("uncertainty").head(50).to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig",
)

experiment_record = {
    "case": "rnn_message_story",
    "question": "只把 SimpleRNN 換成 GRU 後，驗證集 F1 是否改變？",
    "baseline": {"recurrent_layer": "SimpleRNN(32)", "metrics": baseline_metrics},
    "candidate": {"recurrent_layer": "GRU(32)", "metrics": candidate_metrics},
    "single_change": "循環層種類",
    "selected_by_validation_f1": chosen_version,
    "f1_change": float(f1_change),
    "human_review": "低信心、涉及人身安全、來源不明或語意模糊的訊息",
    "limitation": "僅文字分類，未查證事件真實性",
}

record_path = Path("/content/rnn_message_experiment.json")
record_path.write_text(
    json.dumps(experiment_record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(record_path, review_path)


In [ ]:
try:
    from google.colab import files
    files.download(str(record_path))
    files.download(str(review_path))
except ImportError:
    print("檔案已保留在", record_path, "與", review_path)
